**Desafío Data Sintética**

**Introducción**
Este documento define el Desafío Técnico para el rol de Coordinador TDM, combinando una presentación formal del objetivo del área con las especificaciones técnicas del ejercicio.

**Objetivo**
La Unidad de Entornos No Productivos – Chapter QA busca garantizar datos adecuados, reproducibles y trazables para pruebas funcionales y E2E, para esto se solicita generar un dataset sintético de clientes bajo un contrato de datos explícito, validarlo contra reglas de negocio y producir artefactos consumibles con trazabilidad (logs, reportes, perfiles).

In [ ]:
# =====================================
# SETUP DEL ENTORNO
# Instalación de dependencias
# =====================================
!pip install pyspark pyyaml faker -q

print("✅ Librerías instaladas correctamente")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 18.4 MB/s eta 0:00:00
✅ Librerías instaladas correctamente


In [ ]:
# =====================================
# INICIALIZACIÓN DE SPARK
# =====================================
from pyspark.sql import SparkSession


spark = SparkSession.builder \
    .appName("TDM_Challenge") \
    .master("local[*]") \
    .getOrCreate()

print(spark.version)

4.0.2


Generación de 100–500 registros de clientes sintéticos (Parametrizable)

In [12]:
# =====================================
# PARÁMETROS DEL PROCESO
# =====================================

NUM_REGISTROS = 400   # puedes cambiar entre 100–500
PORC_ERROR = 0.0005    # 0.05% porcentaje de errores
# =====================================
# CÁLCULO DE ERRORES
# =====================================

num_errores = max(1, int(NUM_REGISTROS * PORC_ERROR))

print("🚨 Registros con error a inyectar:", num_errores)
print("📊 Registros:", NUM_REGISTROS)
print("⚠️ Porcentaje de error:", PORC_ERROR)

🚨 Registros con error a inyectar: 1
📊 Registros: 400
⚠️ Porcentaje de error: 0.0005


In [11]:
# Crear el YAML corregido y mejorado
yaml_content = """
dataset: clientes_no_productivo
version: 1.0
description: Dataset sintético bancario con catálogos, reglas de calidad y campos sensibles
catalogos:
  estado_cliente:
    - ACTIVO
    - INACTIVO

tables:
  clientes:
    primary_key: customer_id

    columns:

      customer_id:
        type: string
        required: true
        unique: true
        pattern: "^CUS[0-9]{3}$"

      nombre:
        type: string
        required: true
        length: 50
        sensitive: true

      apellido:
        type: string
        required: true
        length: 50
        sensitive: true

      cedula:
        type: string
        required: true
        length: 10
        pattern: "^[0-9]{10}$"
        sensitive: true

      fecha_nacimiento:
        type: date
        required: true
        min_age: 18
        max_age: 90
        format: "dd-MM-yyyy"

      email:
        type: string
        required: true
        format: email
        sensitive: true

      direccion:
        type: string
        required: true
        length: 100
        sensitive: true

      telefono:
        type: string
        required: true
        pattern: "^[0-9]{10}$"
        sensitive: true

      fecha_creacion:
        type: timestamp
        required: true

      estado_cliente:
        type: string
        required: true
        catalog: estado_cliente
"""

with open("contract.yaml", "w", encoding="utf-8") as f:
    f.write(yaml_content)

print("✅ YAML corregido y creado correctamente")

✅ YAML corregido y creado correctamente


In [8]:
import yaml

# Leer el archivo YAML
with open("contract.yaml", "r", encoding="utf-8") as f:
    contract = yaml.safe_load(f)

print("✅ YAML leído correctamente")
print("Dataset:", contract["dataset"])
print("Versión:", contract["version"])
print("Tablas:", list(contract["tables"].keys()))
print("Columnas de clientes:", list(contract["tables"]["clientes"]["columns"].keys()))
print("Catálogos:", list(contract["catalogos"].keys()))

✅ YAML leído correctamente
Dataset: clientes_no_productivo
Versión: 1.0
Tablas: ['clientes']
Columnas de clientes: ['customer_id', 'nombre', 'apellido', 'cedula', 'fecha_nacimiento', 'email', 'direccion', 'telefono', 'fecha_creacion', 'estado_cliente']
Catálogos: ['estado_cliente']


In [9]:
print(contract["tables"]["clientes"]["primary_key"])
print(contract["tables"]["clientes"]["columns"]["customer_id"])
print(contract["tables"]["clientes"]["columns"]["estado_cliente"])

customer_id
{'type': 'string', 'required': True, 'unique': True, 'pattern': '^CUS[0-9]{3}$'}
{'type': 'string', 'required': True, 'catalog': 'estado_cliente'}
